In [ ]:
using EMSuite
using EMSuite.Geometry
using EMSuite.BasisFunctions
using EMSuite.IntegralEquations
using EMSuite.FastAlgorithms.MLFMA
using EMSuite.FastAlgorithms.MLFMA.Level
using EMSuite.FastAlgorithms.MLFMA.Translation
using EMSuite.FastAlgorithms.MLFMA.Interpolation
using LinearAlgebra
using StaticArrays
using SpecialFunctions

function debug_mlfma_factor()
    freq = 1e8
    c0 = 299792458.0
    k = 2π * freq / c0
    eta = 376.73031346177
    
    println("Frequency: $freq")
    println("k: $k")
    println("eta: $eta")
    
    # Define two triangles far apart
    # Tri 1 at origin
    v1 = [0.0 1.0 0.0; 0.0 0.0 1.0; 0.0 0.0 0.0] # Unit triangle in yz plane
    # Tri 2 at (10, 0, 0)
    dist = 10.0
    v2 = [10.0 11.0 10.0; 0.0 0.0 1.0; 0.0 0.0 0.0] # Shifted
    
    # L = 10 (ensure convergence)
    L = 10
    
    # Let's use `EMSuite.FastAlgorithms.MLFMA.Level.levelIntegralInfoCal`
    L_cal, poles = EMSuite.FastAlgorithms.MLFMA.Level.levelIntegralInfoCal(1.0; λ=c0/freq)
    println("Calculated L: $L_cal")
    nPoles = length(poles.r̂sθsϕs)
    
    # Pick a direction (pole)
    iPole = 1
    
    # Tri 1 properties
    l1 = 1.0
    rho1 = [0.0, 1/3, -2/3] # Approx center rho
    
    # Tri 2 properties
    l2 = 1.0
    rho2 = [0.0, 1/3, -2/3] # Local rho same
    r2_center = [dist, 0.0, 0.0]
    
    # 2. Translation (T_kl)
    Z_mlfma = 0.0 + 0.0im
    
    # Translation factor
    # const_factor = -im * k / (16 * π^2)
    const_factor = -im * k / (16 * π^2)
    
    # Translation vector R = r2 - r1 = (10, 0, 0)
    R_vec = [dist, 0.0, 0.0]
    Rab = dist
    R_hat = R_vec / Rab
    
    # Precompute H2
    x = k * Rab
    h2lxs = [SpecialFunctions.sphericalbesselj(l, x) - im * SpecialFunctions.sphericalbessely(l, x) for l in 0:L_cal]
    
    for i in 1:nPoles
        r_hat = poles.r̂sθsϕs[i].r̂
        theta_hat = poles.r̂sθsϕs[i].θhat
        phi_hat = poles.r̂sθsϕs[i].ϕhat
        w = poles.Wθϕs[i]
        
        # Aggregation (Tri 1)
        V1_vec = l1/2 * rho1 # Phase 0
        V1_theta = dot(V1_vec, theta_hat)
        V1_phi = dot(V1_vec, phi_hat)
        
        # Disaggregation (Tri 2)
        phase_rec = exp(-im * k * dot(r_hat, r2_center))
        V2_vec = l2/2 * rho2 * phase_rec
        V2_theta = dot(V2_vec, theta_hat)
        V2_phi = dot(V2_vec, phi_hat)
        
        # Translation Alpha
        cos_phi = dot(r_hat, R_hat)
        # Legendre
        Pl = zeros(L_cal + 1)
        Pl[1] = 1.0
        if L_cal >= 1
            Pl[2] = cos_phi
        end
        for l in 1:L_cal-1
            Pl[l+2] = ((2l+1)*cos_phi*Pl[l+1] - l*Pl[l])/(l+1)
        end
        
        alpha = 0.0 + 0.0im
        j_term = 1.0 + 0.0im
        for l in 0:L_cal
            term_val = j_term * (2l+1) * h2lxs[l+1] * Pl[l+1]
            alpha += term_val
            j_term *= -im
        end
        alpha *= const_factor * w
        
        term = (V1_theta * V2_theta + V1_phi * V2_phi) * alpha
        Z_mlfma += term
    end
    
    # Apply Disagg factor
    Z_mlfma *= (im * k * eta)
    
    println("MLFMA Z: $Z_mlfma")
    println("MLFMA |Z|: $(abs(Z_mlfma))")
    
    # 3. Direct
    # Z_direct = j k eta / 16pi * l1 * l2 * (rho1 . rho2 * G)
    
    G = exp(-im * k * dist) / dist
    dot_rho = dot(rho1, rho2)
    
    Z_direct = (im * k * eta / (16 * π)) * l1 * l2 * (dot_rho * G)
    
    println("Direct Z: $Z_direct")
    println("Direct |Z|: $(abs(Z_direct))")
    
    ratio = abs(Z_mlfma) / abs(Z_direct)
    println("Ratio (MLFMA/Direct): $ratio")
    println("Ratio dB: $(20log10(ratio))")
end

debug_mlfma_factor()